## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [5]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [6]:
MODEL = "openai/gpt-oss-120b"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [12]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [13]:
retriever = vectorstore.as_retriever()
llm = ChatGroq(temperature=0, model_name=MODEL, reasoning_effort="low")

### These LangChain objects implement the method `invoke()`

In [14]:
retriever.invoke("Who is Avery?")

[Document(id='d1934f5d-609f-4b14-9937-2f4cee3bb701', metadata={'doc_type': 'employees', 'source': 'knowledge-base\\employees\\Avery Lancaster.md'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and a

In [15]:
llm.invoke("Who is Avery?")

AIMessage(content='I’m not sure which\u202fAvery you’re referring to. Could you give me a bit more context—perhaps a last name, a field (e.g., literature, science, entertainment), or where you heard the name? That’ll help me give you the most accurate answer.', additional_kwargs={'reasoning_content': 'User asks "Who is Avery?" Probably ambiguous. Need ask clarification.'}, response_metadata={'token_usage': {'completion_tokens': 81, 'prompt_tokens': 75, 'total_tokens': 156, 'completion_time': 0.171419802, 'completion_tokens_details': {'reasoning_tokens': 15}, 'prompt_time': 0.013578534, 'prompt_tokens_details': None, 'queue_time': 0.376563063, 'total_time': 0.184998336}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c868cf1eaa', 'service_tier': 'on_demand', 'reasoning_effort': 'low', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1d5f2-3796-7467-be2e-2ed6f1e3724e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_token

## Time to put this together!

In [16]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [17]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [18]:
answer_question("Who is Averi Lancaster?", [])

'**Avery Lancaster** is one of the founding leaders of Insurellm.\n\n- **Title:** Co‑Founder & Chief Executive Officer (CEO)  \n- **Location:** San Francisco, California  \n- **Date of Birth:** March\u202f15,\u202f1985  \n- **Current Salary:** $225,000  \n\n### Career at Insurellm\n- **2015\u202f–\u202fPresent:** Co‑Founder & CEO  \n  - Since co‑founding Insurellm in 2015, Avery has steered the company to become a leading Insurance‑Tech provider. She is recognized for innovative leadership, strong risk‑management expertise, and the ability to bring cutting‑edge insurance products to market.\n\n### Prior Experience\n- **2013\u202f–\u202f2015:** Senior Product Manager at Innovate Insurance Solutions  \n  - Developed groundbreaking insurance products for the technology sector, laying the groundwork for the vision she later brought to Insurellm.\n\nAvery’s strategic direction and entrepreneurial spirit continue to drive Insurellm’s growth and industry impact. If you’d like more details abo

## What could possibly come next? 😂

In [19]:
gr.ChatInterface(answer_question).launch()

c:\Users\achus\Documents\llm-engineering\.venv\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


## Admit it - you thought RAG would be more complicated than that!!